# KDE Dataset Extraction: Convective Environment Variables by Lightning/OT Category

Purpose: Extract full (point-level) distributions of selected convective environment
variables (MUCAPE, MULCL, RH36mean) for each lightning/OT co-occurrence category (OT only,
OT+LS, LS only), separately for land and ocean, to support kernel density estimate (KDE)
plots of their distributions across warm seasons (2016–2024).

Input:
- Categorized lightning/OT point CSVs (from 06_categorical_lightning_OT_dilation.ipynb):
  ls_only_land_3d.csv, ls_only_ocean_3d.csv, ls_ot_land_3d.csv, ls_ot_ocean_3d.csv,
  ot_only_land_3d.csv, ot_only_ocean_3d.csv
- BARRA-R2 hourly reanalysis variables: MUCAPE, MULCL, RH36mean
- Region of interest: Northern Australia (lat: -20 to -10, lon: 120 to 148, +2° pad for
  file loading)
- Warm season: October-April, years 2016-2024

Processing:
1. Load categorized lightning/OT CSVs and group into three categories (OT only, OT+LS,
   LS only), each split by land/ocean.
2. For each variable, load matching monthly NetCDF files for warm-season months across
   all years, spatially subsetting with a buffer, and mask to convective conditions
   using MUCAPE > 0.
3. For each category/contrast combination, extract full point-level data at the nearest
   grid point/time matching each categorized lightning/OT detection.
4. Concatenate all years/categories into one dataframe per variable and save to feather
   for downstream KDE plotting.

Output:
- Per-variable point-level dataframes:
  /g/data/if69/tp4064/Project_1/categorical_position/Box_plot/KDE_dataset_<var>.feather

In [1]:
# Standard libraries
import os
import glob

# Data manipulation
import pandas as pd
import numpy as np
import xarray as xr


# Parallel computing
import dask
from dask.distributed import Client

# Initialize Dask client with one thread per worker
client = Client(threads_per_worker=1)
# client = Client()
client



In [17]:
# ── Helper: match monthly NetCDF files in a date range ────────────────────────
def get_files(path: str, start: str, end: str, warm_months) -> list[str]:
    months = pd.date_range(start=start, end=end, freq='MS')#.strftime('%Y%m')
    months = months[months.month.isin(warm_months)].strftime('%Y%m')
    files  = sorted(f for m in months for f in glob.glob(os.path.join(path, f'*-{m}.nc')))
    return files

# ──
## Filter Data based on input given, specifically design for Land/Ocean Region for OT.
#This function filters the dataset to a specific lat/lon box and removes data from 2015.

def select_point_based_data(data, variable,year, warm_months=(10, 11, 12, 1, 2, 3, 4)):
    """
    Select nearest grid-point data for OT/lightning points.
    """

    # Copy to avoid modifying original dataframe
    variable = variable.copy()

    # Convert time once
    variable['time'] = pd.to_datetime(variable['time'])

    # Single combined mask (faster + cleaner)
    mask = (
        (variable['lat'].between(LAT_MIN, LAT_MAX)) &
        (variable['lon'].between(LON_MIN, LON_MAX)) &
        (variable['time'].dt.year == year) &
        (variable['time'].dt.month.isin(warm_months))
    )

    variable = variable.loc[mask]

    # Reduce precision slightly for cleaner nearest-neighbour matching
    variable['lat'] = variable['lat'].round(4)
    variable['lon'] = variable['lon'].round(4)

    # Convert directly to NumPy arrays (lighter than pandas Series)
    da_lat = xr.DataArray(variable['lat'].to_numpy(), dims='points')
    da_lon = xr.DataArray(variable['lon'].to_numpy(), dims='points')
    da_time = xr.DataArray(variable['time'].to_numpy(), dims='points')

    # Nearest selection
    data_select = data.sel(
        lat=da_lat,
        lon=da_lon,
        time=da_time,
        method='nearest'
    )

    return data_select


def preprocess_fn(ds):
    PAD = 2
    ds = (
        ds.drop_vars("crs", errors="ignore")
          .sel(
              lat=slice(LAT_MIN - PAD, LAT_MAX + PAD),
              lon=slice(LON_MIN - PAD, LON_MAX + PAD),
              # time=slice(, END_DATE),    # ← add back here
          )
    )

    # Filter warm months (Oct–Apr) per file before concat
    # warm_mask = ds.time.dt.month.isin(warm_months)
    # ds = ds.isel(time=warm_mask)
    
    return ds



def yearly_data_processing(years, DATA_DIR, var, CATEGORIES, warm_months=[10, 11, 12, 1, 2, 3, 4]):

    all_data = []
    
    for year in years:
        start = f"{year}-01-01"
        end   = f"{year}-12-31"
        print(f"Processing: {start} to {end}")

        # 1. Load Main Dataset
        df_all = xr.open_mfdataset(
            get_files(
                os.path.join(DATA_DIR, var, 'latest/'),
                start,
                end,
                warm_months=warm_months # Use parameter, not hardcoded list
            ),
            chunks={},  
            combine='by_coords',
            preprocess=preprocess_fn
        )
        
        # 2. Load Mask Dataset separately so we can close it later
        ds_mucape = xr.open_mfdataset(
            get_files(
                os.path.join(DATA_DIR, 'MUCAPE', 'latest/'),
                start,
                end,
                warm_months=warm_months
            ),
            chunks={},  
            combine='by_coords',
            preprocess=preprocess_fn
        )
        
        data_mucape_mask = ds_mucape['MUCAPE'] > 0

        # Apply lazy mask
        df_masked = df_all.where(data_mucape_mask)

        # 3. Process Categories
        for group, (land_df, ocean_df) in CATEGORIES.items():
            for contrast, cat_df in [('Land', land_df), ('Ocean', ocean_df)]:
                
                # Select points (still a lazy dask graph at this point)
                dd = select_point_based_data(df_masked, cat_df, year, warm_months=warm_months)

                # Compute and convert to Pandas
                dd_df = dd.to_dataframe()
                dd_df['group'] = group
                dd_df['contrast'] = contrast
                
                # Append directly to the master list
                all_data.append(dd_df)

        # 4. Cleanup memory/file-handlers before the next year starts
        df_all.close()
        ds_mucape.close()

    # 5. Concatenate ONLY once at the very end
    df = pd.concat(all_data)

    return df

In [23]:
# Input data
list_dir_1 = ['MUCAPE', 'MULCL', 'RH36mean']
LAT_MIN, LAT_MAX = -20, -10
LON_MIN, LON_MAX = 120, 148
DATA_DIR   = '/g/data/ob53/BARRA2/output/reanalysis/AUST-11/BOM/ERA5/historical/hres/BARRA-R2/v1/1hr/'
PAD = 2
warm_months = [10, 11, 12, 1, 2, 3, 4]
years = range(2016, 2025)
CSV_DIR = "/g/data/if69/tp4064/Project_1/categorical_position/Dialation/"

csv_files = {
    "ls_only_land"   : "ls_only_land_3d.csv",
    "ls_only_ocean"  : "ls_only_ocean_3d.csv",
    "ls_ot_land"     : "ls_ot_land_3d.csv",
    "ls_ot_ocean"    : "ls_ot_ocean_3d.csv",
    "land_ot_no_ls"  : "ot_only_land_3d.csv",
    "ocean_ot_no_ls" : "ot_only_ocean_3d.csv",
}

In [24]:
# Processing data fro KDE plot
data = {name: pd.read_csv(CSV_DIR + file) for name, file in csv_files.items()}

CATEGORIES = {
    'OT only': (data['land_ot_no_ls'],  data['ocean_ot_no_ls']),
    'OT + LS': (data['ls_ot_land'],     data['ls_ot_ocean']),
    'LS only': (data['ls_only_land'],   data['ls_only_ocean']),}


all_stats = []

for var in list_dir_1 :

    print(var)

    df = yearly_data_processing(years, DATA_DIR, var, CATEGORIES, warm_months=[10, 11, 12, 1, 2, 3, 4])

    df.to_feather(f"/g/data/if69/tp4064/Project_1/categorical_position/Box_plot/1_KDE_dataset_{var}.feather")

    # stats = compute_stats(df)
    # stats['variable'] = var
    
    # all_stats.append(stats)
        

MUCAPE
Processing: 2016-01-01 to 2016-12-31
Processing: 2017-01-01 to 2017-12-31
Processing: 2018-01-01 to 2018-12-31
Processing: 2019-01-01 to 2019-12-31
Processing: 2020-01-01 to 2020-12-31
Processing: 2021-01-01 to 2021-12-31
Processing: 2022-01-01 to 2022-12-31
Processing: 2023-01-01 to 2023-12-31
Processing: 2024-01-01 to 2024-12-31
MULCL
Processing: 2016-01-01 to 2016-12-31
Processing: 2017-01-01 to 2017-12-31


/g/data/xp65/public/apps/med_conda/envs/analysis3-26.04/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 10.69 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Processing: 2018-01-01 to 2018-12-31
Processing: 2019-01-01 to 2019-12-31


/g/data/xp65/public/apps/med_conda/envs/analysis3-26.04/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 11.57 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Processing: 2020-01-01 to 2020-12-31
Processing: 2021-01-01 to 2021-12-31


...

Processing: 2022-01-01 to 2022-12-31


/g/data/xp65/public/apps/med_conda/envs/analysis3-26.04/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 10.47 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Processing: 2023-01-01 to 2023-12-31


/g/data/xp65/public/apps/med_conda/envs/analysis3-26.04/lib/python3.12/site-packages/distributed/client.py:3387: UserWarning: Sending large graph of size 10.84 MiB.
This may cause some slowdown.
Consider loading the data with Dask directly
 or using futures or delayed objects to embed the data into the graph without repetition.
See also https://docs.dask.org/en/stable/best-practices.html#load-data-with-dask for more information.
  warnings.warn(


Processing: 2024-01-01 to 2024-12-31


# THE ABOVE CODE, is calculation for dataset for KDE warm season